<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/banners/banner_customer_churn_powerbi_guide.png" width="100%"/>
</div>

## 📖 Préambule

### À qui s'adresse ce guide ?

Ce notebook est ton **support de référence** pour construire le tableau de bord *Customer Churn Analytics — IvoirCom* dans Power BI Desktop. C'est un dashboard de pilotage commercial pour un opérateur télécom mobile en Côte d'Ivoire.

### Comment lire ce guide

| Symbole | Ce qu'il indique |
|---|---|
| 🎯 | Ce que tu sauras faire à la fin de la section |
| 📘 | L'intuition métier ou technique avant de coder |
| 🔧 | Les clics, le code DAX, les paramètres exacts |
| 🎓 | Une méthode opérationnelle pour construire un visuel précis |
| ✅ | Comment vérifier que ton travail est correct |
| ⚠️ | L'erreur courante à éviter |

### Le contexte métier

**IvoirCom** est un opérateur télécom mobile fictif basé à Abidjan, opérant sur 5 villes ivoiriennes (Abidjan, Bouaké, Yamoussoukro, San-Pédro, Korhogo) avec 6 offres au catalogue. La direction commerciale alerte : **12 % de la base abonnés churn chaque trimestre** (25,4 % cumulatif sur 24 mois). Le dashboard pilote 4 enjeux :

1. **Taux de churn** — global et par segment (offre, ville, tranche d'âge), cible interne < 20 %
2. **ARPU** — moyenne par client actif vs churners, pour mesurer la valeur perdue
3. **Réclamations** — signal avant-coureur quand 2+ tickets ne sont pas résolus (93,4 % de churn dans ce segment)
4. **Plan d'action** — segmentation RFM Telecom à 6 segments, top 50 At Risk à recontacter

Le dashboard répond à 5 questions :

| Page | Question |
|---|---|
| 1 — Vue Executive | Quelle est la santé globale de la base abonnés IvoirCom ? |
| 2 — Segments à risque | Qui churne le plus (par offre, ville, âge) ? |
| 3 — Cohortes & Rétention | Quelle génération de souscripteurs tient le mieux ? |
| 4 — Signaux Réclamations | Les plaintes prédisent-elles le départ ? |
| 5 — Plan d'action | Quels clients appeler aujourd'hui (At Risk Top 50) ? |

---
# I — Préparer les fondations

## 1.1 Comprendre les sources de données

### 📘 Concept clé — 5 tables sources + 1 table dérivée RFM

Le projet utilise **5 tables sources** issues du système d'information IvoirCom (CRM + billing + helpdesk) :

- `clients` — 8 000 abonnés propres (après nettoyage 30 doublons + 5 âges négatifs) avec offre, ville, dates de souscription / résiliation
- `offres` — 6 offres au catalogue (Pulse, Connect, Premium, Pro, Étudiant, Senior) avec prix et inclus
- `factures` — 138 284 factures mensuelles sur 24 mois (2 % de montants nuls à exclure de l'ARPU)
- `consommation_mensuelle` — 137 044 lignes voix / SMS / data par client par mois
- `reclamations` — 9 791 tickets support (3 % de délais négatifs, 30 % en statut `ouvert`)

Et **1 table dérivée** produite par le notebook SQL :

- `clients_rfm` — 1 ligne par client avec scores R / F / M (NTILE 1-5) et segment final (Champions, Loyal, At Risk, Lost, New, Others)

### ⚠️ Piège fréquent — quelle table pour quel KPI ?

- **KPIs volume / churn / âge / ville / offre** ⇒ `clients` (table de référence)
- **ARPU / CA / méthode de paiement** ⇒ `factures`
- **Tickets / délai / type de plainte** ⇒ `reclamations`
- **Voix / SMS / data** ⇒ `consommation_mensuelle`
- **Segment RFM / Top At Risk** ⇒ `clients_rfm` (sortie du notebook SQL)

## 1.2 Importer les CSV

### 📘 Concept clé — pourquoi GitHub raw plutôt que des fichiers locaux ?

Charger depuis une URL `raw.githubusercontent.com` te donne deux superpouvoirs :
1. **Reproductibilité** : tous les apprenants ont exactement la même donnée, à l'octet près.
2. **Mise à jour facile** : si on corrige une coquille dans le CSV, un simple *Actualiser* suffit, aucune ré-installation.

L'inconvénient : il faut une connexion internet au premier chargement. Une fois publié sur le service Power BI, le rapport peut être planifié pour rafraîchir tout seul.

### Les 6 URLs à utiliser

```
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_churn_analytics/dataset/clients.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_churn_analytics/dataset/offres.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_churn_analytics/dataset/factures.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_churn_analytics/dataset/consommation_mensuelle.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_churn_analytics/dataset/reclamations.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_churn_analytics/dataset/clients_rfm.csv
```

> 💡 *Si `clients_rfm.csv` n'existe pas encore : exécuter le notebook SQL puis ajouter à la fin une cellule `%%sql COPY rfm TO '../../dataset/clients_rfm.csv' (HEADER, DELIMITER ',');`*

### 🔧 Procédure pas-à-pas

Pour chacun des 6 CSV : **Accueil → Obtenir les données → Web → Coller l'URL → OK → Charger**.

Dans le panneau **Power Query** :
- Vérifier que les colonnes `date_*` sont bien typées **Date** (pas Texte).
- Sur `factures[montant_fcfa]` et `clients[age]` : type **Nombre entier**.
- Sur `reclamations[delai_resolution_jours]` : type **Nombre entier** (les NULL sont conservés tels quels).
- Renommer la requête `consommation_mensuelle` en `consommation` (plus court).

## 1.3 Désactiver l'Auto Date/Time

**Fichier → Options → Chargement des données → décocher Date/heure automatique**.

Sans ça, Power BI crée une LocalDateTable cachée pour chaque colonne de date — sur ce projet, c'est 5 tables fantômes en moins (date_facturation, date_souscription, date_resiliation, date_creation, mois).

---
# II — Modéliser les données

## 2.1 Schéma en étoile

### 📘 Concept clé — `clients` est la table pivot, les fact tables sont satellites de Calendrier

```
                  +------------------+
                  |   Calendrier     |   <-- table de dates (DAX)
                  +--------+---------+
                           | 1 (4 relations vers chaque table date)
                           v
    +--------+ N        N  +------+         N    +------------+
    | offres +-----[1]----+ clients +---[1]----+ clients_rfm |
    +--------+              +-+-+-+              +------------+
                              | | |
                              | | +---[1]--N--> consommation
                              | +---[1]--N--> reclamations
                              +---[1]--N--> factures
```

## 2.2 Créer la table Calendrier

**Modélisation → Nouvelle table** :

```dax
Calendrier = 
ADDCOLUMNS(
    CALENDAR(DATE(2023,1,1), DATE(2025,12,31)),
    "Annee",         YEAR([Date]),
    "Mois_Num",      MONTH([Date]),
    "Mois_Nom",      FORMAT([Date], "mmm", "fr-FR"),
    "Annee_Mois",    FORMAT([Date], "yyyy-MM"),
    "Trimestre",     "T" & QUARTER([Date]),
    "Jour_Semaine",  FORMAT([Date], "dddd", "fr-FR")
)
```

## 2.3 Marquer Calendrier comme table de dates

Vue Données → `Calendrier` → **Outils de table → Marquer comme table de dates → colonne Date**.

## 2.4 Établir les relations

| # | De (1) | Clé | Vers (N) | Clé | État |
|---|---|---|---|---|---|
| 1 | `Calendrier` | `Date` | `factures` | `date_facturation` | Active |
| 2 | `Calendrier` | `Date` | `reclamations` | `date_creation` | Active |
| 3 | `Calendrier` | `Date` | `consommation` | `mois` | Active |
| 4 | `Calendrier` | `Date` | `clients` | `date_souscription` | **Inactive** |
| 5 | `Calendrier` | `Date` | `clients` | `date_resiliation` | **Inactive** |
| 6 | `offres` | `code_offre` | `clients` | `code_offre` | Active |
| 7 | `clients` | `id_client` | `factures` | `id_client` | Active |
| 8 | `clients` | `id_client` | `reclamations` | `id_client` | Active |
| 9 | `clients` | `id_client` | `consommation` | `id_client` | Active |
| 10 | `clients` | `id_client` | `clients_rfm` | `id_client` | Active (1:1) |

### ⚠️ Pourquoi 2 relations Calendrier ↔ clients en INACTIVE ?

Power BI n'autorise **qu'une seule relation active** entre deux tables. On doit donc choisir : `date_souscription` ou `date_resiliation` ? Aucune ne s'impose. Solution : on les laisse **inactives** et on les active à la demande dans une mesure via `USERELATIONSHIP()`. Exemple :

```dax
Souscriptions Mensuelles = 
CALCULATE(
    COUNTROWS(clients),
    USERELATIONSHIP(clients[date_souscription], Calendrier[Date])
)
```

## 2.5 Colonnes calculées dans `clients`

Quatre colonnes enrichissent `clients` pour faciliter les segmentations :

```dax
-- Colonne 1 : flag churner (0 ou 1) — utile pour SUM rapide
Est_Churner = IF(clients[statut] = "resilie", 1, 0)

-- Colonne 2 : tranche d'age metier (5 buckets)
Tranche_Age = 
SWITCH(TRUE(),
    clients[age] >= 18 && clients[age] <= 25, "1. 18-25",
    clients[age] >= 26 && clients[age] <= 35, "2. 26-35",
    clients[age] >= 36 && clients[age] <= 45, "3. 36-45",
    clients[age] >= 46 && clients[age] <= 60, "4. 46-60",
    "5. 60+"
)

-- Colonne 3 : mois de cohorte (1er du mois de souscription)
Cohorte_Mois = DATE(YEAR(clients[date_souscription]), MONTH(clients[date_souscription]), 1)

-- Colonne 4 : anciennete en mois (date pivot 2025-12-31 si actif)
Anciennete_Mois = 
DATEDIFF(
    clients[date_souscription],
    IF(ISBLANK(clients[date_resiliation]), DATE(2025,12,31), clients[date_resiliation]),
    MONTH
)
```

---
# III — Créer la table `_Mesures`

**Accueil → Entrer des données →** 1 colonne, 1 ligne vide → nommer `_Mesures` → **Charger**.

Après avoir créé ta première mesure et l'avoir glissée dans `_Mesures`, supprime la colonne fictive.

---
# IV — Construire les ~50 mesures DAX

### Vue d'ensemble des dossiers

| # | Dossier | Mesures | Rôle |
|---|---|---|---|
| 0 | Pilotage Opérationnel | 6 | Sous-titres dynamiques + bandeau alerte signal |
| 1 | KPIs Globaux | 9 | Total Clients, Taux Churn %, ARPU global, écart actifs/churners |
| 2 | Évolution | 5 | Souscriptions / résiliations mensuelles avec USERELATIONSHIP |
| 3 | Vue Segmentation | 7 | Taux churn par offre, ville, tranche âge + rangs |
| 4 | Vue Cohortes & Ancienneté | 5 | Anciennetés moyennes, rétention M+3 / M+6 / M+12 |
| 5 | Vue Réclamations | 8 | Total tickets, % résolus / ouverts / abandonnés, délai moyen |
| 6 | Vue Plan Action | 8 | Nb 2+ tickets non résolus, taux churn segment, segments RFM |
| _ | _Helpers | 4 | Mesures couleur dynamiques |

## 4.1 Dossier `0. Pilotage Opérationnel` (6 mesures)

```dax
Sous Titre Vue Executive = 
"Etat de la base " & FORMAT([Total Clients], "#,##0") & 
" abonnes  -  Taux churn cumulatif " & FORMAT([Taux Churn %], "0.0") & " %"

Sous Titre Segments = 
"Top offre a risque : Pulse " & FORMAT(35.3, "0.0") & 
" %  -  " & [Nb Offres Au Dessus Seuil] & " offres > 25 % de churn"

Sous Titre Cohortes = 
"Anciennete moyenne actifs " & FORMAT([Anciennete Actifs Mois], "0.0") & 
" mois  vs  churners " & FORMAT([Anciennete Churners Mois], "0.0") & " mois"

Sous Titre Reclamations = 
FORMAT([Total Reclamations], "#,##0") & " tickets  -  " & 
FORMAT([Pct Tickets Non Resolus], "0.0") & " % non resolus (ouvert + abandonne)"

Sous Titre Plan Action = 
"At Risk : " & FORMAT([Nb Clients At Risk], "#,##0") & 
"  -  Top 50 = " & FORMAT([ARPU Top 50 At Risk], "#,##0") & " FCFA / an exposes"

Bandeau Alerte = 
VAR _n = [Nb Clients 2 Plus Tickets Non Resolus]
VAR _taux = [Taux Churn Segment Risque %]
RETURN IF(
    _n > 0,
    "⚠ " & FORMAT(_n, "#,##0") & " clients a 2+ tickets non resolus  -  Taux de churn observe " & 
        FORMAT(_taux, "0.0") & " %  -  Action immediate requise",
    "✓ Aucun signal critique en cours - Niveau de service nominal"
)
```

### 📘 Pourquoi des sous-titres dynamiques ?

Au lieu d'écrire « Vue Executive » en dur en haut de page, on affiche un **sous-titre qui se met à jour avec les filtres**. Quand l'utilisateur active le slicer Année 2025, le sous-titre recalcule automatiquement le taux de churn 2025 — c'est ce qui transforme un dashboard statique en outil d'investigation.

## 4.2 Dossier `1. KPIs Globaux` (9 mesures)

```dax
Total Clients = COUNTROWS(clients)

Total Actifs = CALCULATE(COUNTROWS(clients), clients[statut] = "actif")

Total Resilies = CALCULATE(COUNTROWS(clients), clients[statut] = "resilie")

Taux Churn % = 
DIVIDE(
    [Total Resilies],
    [Total Clients]
) * 100

Taux Churn Ratio = DIVIDE([Total Resilies], [Total Clients])

Taux Conforme Ratio = 1 - [Taux Churn Ratio]

ARPU Mensuel Moyen = 
AVERAGEX(
    SUMMARIZE(
        FILTER(factures, factures[montant_fcfa] > 0),
        factures[id_client]
    ),
    CALCULATE(AVERAGE(factures[montant_fcfa]))
)

ARPU Actifs = 
CALCULATE([ARPU Mensuel Moyen], clients[statut] = "actif")

ARPU Churners = 
CALCULATE([ARPU Mensuel Moyen], clients[statut] = "resilie")
```

### ⚠️ Pourquoi `Taux Churn %` ET `Taux Churn Ratio` ?

- `Taux Churn %` retourne **25,4** (déjà × 100, format `0.0\"%\"`) — pour cards et bandeau
- `Taux Churn Ratio` retourne **0,254** (ratio 0-1, format Pourcentage) — pour les barres empilées 100 %

Cette dualité existe car certains visuels Power BI attendent un ratio (les barres empilées 100 %), d'autres préfèrent un nombre déjà formaté.

## 4.3 Dossier `2. Évolution` (5 mesures)

```dax
Souscriptions Mensuelles = 
CALCULATE(
    COUNTROWS(clients),
    USERELATIONSHIP(clients[date_souscription], Calendrier[Date])
)

Resiliations Mensuelles = 
CALCULATE(
    COUNTROWS(clients),
    USERELATIONSHIP(clients[date_resiliation], Calendrier[Date]),
    NOT(ISBLANK(clients[date_resiliation]))
)

Churn Net Mois = [Resiliations Mensuelles] - [Souscriptions Mensuelles]

Souscriptions YTD = 
TOTALYTD(
    [Souscriptions Mensuelles],
    Calendrier[Date]
)

Resiliations YTD = 
TOTALYTD(
    [Resiliations Mensuelles],
    Calendrier[Date]
)
```

### 📘 USERELATIONSHIP — pourquoi 2 relations inactives ?

Le même `Calendrier[Date]` doit pouvoir filtrer **soit** les souscriptions, **soit** les résiliations selon le contexte. Power BI n'autorise qu'**une relation active** entre deux tables : on garde donc les 2 relations inactives et on choisit dynamiquement avec `USERELATIONSHIP()` à l'intérieur de chaque mesure.

## 4.4 Dossier `3. Vue Segmentation` (7 mesures)

```dax
Nb Clients Offre = COUNTROWS(clients)

Taux Churn Offre % = 
DIVIDE(
    CALCULATE(COUNTROWS(clients), clients[statut] = "resilie"),
    COUNTROWS(clients)
) * 100

Rang Offre Churn = 
RANKX(
    ALL(offres[code_offre]),
    CALCULATE([Taux Churn Offre %]),
    ,
    DESC,
    DENSE
)

Nb Offres Au Dessus Seuil = 
VAR _seuil = 25
RETURN
COUNTROWS(
    FILTER(
        VALUES(offres[code_offre]),
        CALCULATE([Taux Churn Offre %]) > _seuil
    )
)

Taux Churn Ville % = 
DIVIDE(
    CALCULATE(COUNTROWS(clients), clients[statut] = "resilie"),
    COUNTROWS(clients)
) * 100

Taux Churn Tranche Age % = 
DIVIDE(
    CALCULATE(COUNTROWS(clients), clients[statut] = "resilie"),
    COUNTROWS(clients)
) * 100

Top Ville Churn = 
VAR _t = 
    TOPN(1,
        ADDCOLUMNS(
            VALUES(clients[ville]),
            "@taux", CALCULATE([Taux Churn %])
        ),
        [@taux],
        DESC
    )
RETURN
MAXX(_t, clients[ville]) & " (" & FORMAT(MAXX(_t, [@taux]), "0.0") & " %)"
```

### ⚠️ Piège — `RANKX` qui renvoie 1 partout

Sur un visuel table, sans le `CALCULATE([Taux Churn Offre %])` interne, `RANKX` n'arrive pas à transitionner du contexte de ligne au contexte de filtre. Tous les rangs renvoient 1. **Toujours wrapper la mesure dans CALCULATE quand RANKX itère sur ALL.**

## 4.5 Dossier `4. Vue Cohortes & Ancienneté` (5 mesures)

```dax
Anciennete Mois Moyenne = AVERAGE(clients[Anciennete_Mois])

Anciennete Actifs Mois = 
CALCULATE([Anciennete Mois Moyenne], clients[statut] = "actif")

Anciennete Churners Mois = 
CALCULATE([Anciennete Mois Moyenne], clients[statut] = "resilie")

Retention Cohorte Pct = 
VAR _cohorte_size = 
    CALCULATE(
        COUNTROWS(clients),
        ALLEXCEPT(clients, clients[Cohorte_Mois])
    )
VAR _actifs_cohorte = 
    CALCULATE(
        COUNTROWS(clients),
        clients[statut] = "actif",
        ALLEXCEPT(clients, clients[Cohorte_Mois])
    )
RETURN
DIVIDE(_actifs_cohorte, _cohorte_size) * 100

Pct Cohortes Saines = 
-- Pct des cohortes 2024 ayant > 80 % de retention
VAR _toutes_cohortes = 
    FILTER(
        SUMMARIZE(clients, clients[Cohorte_Mois]),
        YEAR(clients[Cohorte_Mois]) = 2024
    )
VAR _saines = 
    FILTER(_toutes_cohortes, [Retention Cohorte Pct] >= 80)
RETURN
DIVIDE(COUNTROWS(_saines), COUNTROWS(_toutes_cohortes)) * 100
```

### 📘 Construction de la heatmap rétention

On utilisera un visuel **Matrix** (page 3) avec :
- **Lignes** : `clients[Cohorte_Mois]` (formaté yyyy-MM, filtré 2024)
- **Colonnes** : `clients[Anciennete_Mois]` (filtré 0 à 11)
- **Valeurs** : `[Retention Cohorte Pct]`
- **Mise en forme conditionnelle** : dégradé rouge (50%) → vert (100%)

## 4.6 Dossier `5. Vue Réclamations` (8 mesures)

```dax
Total Reclamations = COUNTROWS(reclamations)

Nb Reclamations Resolues = 
CALCULATE(COUNTROWS(reclamations), reclamations[statut] = "resolu")

Nb Reclamations Ouvertes = 
CALCULATE(COUNTROWS(reclamations), reclamations[statut] = "ouvert")

Nb Reclamations Abandonnees = 
CALCULATE(COUNTROWS(reclamations), reclamations[statut] = "abandonne")

Pct Tickets Resolus = 
DIVIDE([Nb Reclamations Resolues], [Total Reclamations]) * 100

Pct Tickets Non Resolus = 
DIVIDE(
    [Nb Reclamations Ouvertes] + [Nb Reclamations Abandonnees],
    [Total Reclamations]
) * 100

Delai Resolution Moyen Jours = 
CALCULATE(
    AVERAGE(reclamations[delai_resolution_jours]),
    reclamations[statut] = "resolu"
)

Tickets Moyens Par Client = 
DIVIDE(
    [Total Reclamations],
    DISTINCTCOUNT(reclamations[id_client])
)
```

### 🎓 MÉTHODE — Comparer tickets churners vs actifs

Pour comparer le nb de tickets entre les deux populations, on **ne peut pas** simplement faire un `AVERAGEX(GROUPBY(...))` à cause du contexte. La bonne approche :

```dax
Tickets Moyens Churners = 
VAR _t = 
    SUMMARIZE(
        FILTER(clients, clients[statut] = "resilie"),
        clients[id_client],
        "@n", CALCULATE(COUNTROWS(reclamations))
    )
RETURN AVERAGEX(_t, [@n] + 0)

Tickets Moyens Actifs = 
VAR _t = 
    SUMMARIZE(
        FILTER(clients, clients[statut] = "actif"),
        clients[id_client],
        "@n", CALCULATE(COUNTROWS(reclamations))
    )
RETURN AVERAGEX(_t, [@n] + 0)

Ratio Tickets Churners vs Actifs = 
DIVIDE([Tickets Moyens Churners], [Tickets Moyens Actifs])
```

Le `+ 0` après `[@n]` force un 0 au lieu de BLANK pour les clients sans aucun ticket — sinon la moyenne biaise vers le haut.

## 4.7 Dossier `6. Vue Plan Action` (8 mesures)

```dax
Nb Clients 2 Plus Tickets Non Resolus = 
VAR _segments = 
    FILTER(
        SUMMARIZE(
            FILTER(reclamations, reclamations[statut] IN {"ouvert", "abandonne"}),
            reclamations[id_client],
            "@n", COUNT(reclamations[id_ticket])
        ),
        [@n] >= 2
    )
RETURN COUNTROWS(_segments)

Taux Churn Segment Risque % = 
VAR _ids_a_risque = 
    SELECTCOLUMNS(
        FILTER(
            SUMMARIZE(
                FILTER(reclamations, reclamations[statut] IN {"ouvert", "abandonne"}),
                reclamations[id_client],
                "@n", COUNT(reclamations[id_ticket])
            ),
            [@n] >= 2
        ),
        "id_client", reclamations[id_client]
    )
VAR _total_segment = COUNTROWS(_ids_a_risque)
VAR _churners_segment = 
    CALCULATE(
        COUNTROWS(clients),
        clients[statut] = "resilie",
        TREATAS(_ids_a_risque, clients[id_client])
    )
RETURN DIVIDE(_churners_segment, _total_segment) * 100

Ratio Signal vs Base = 
DIVIDE([Taux Churn Segment Risque %], [Taux Churn %])

-- Mesures issues de la table clients_rfm (importee depuis le notebook SQL)
Nb Clients Champions = 
CALCULATE(COUNTROWS(clients_rfm), clients_rfm[segment_rfm] = "Champions")

Nb Clients At Risk = 
CALCULATE(COUNTROWS(clients_rfm), clients_rfm[segment_rfm] = "At Risk")

Nb Clients Lost = 
CALCULATE(COUNTROWS(clients_rfm), clients_rfm[segment_rfm] = "Lost")

ARPU Top 50 At Risk = 
VAR _top50 = 
    TOPN(
        50,
        FILTER(clients_rfm, clients_rfm[segment_rfm] = "At Risk"),
        clients_rfm[arpu_6m],
        DESC
    )
RETURN SUMX(_top50, clients_rfm[arpu_6m]) * 12

Pct Champions = 
DIVIDE([Nb Clients Champions], COUNTROWS(clients_rfm)) * 100
```

### 📘 TREATAS pour le signal 2+ tickets non résolus

`TREATAS` permet d'**injecter une liste virtuelle de `id_client`** (calculée depuis reclamations) comme filtre sur la table `clients`. C'est le seul moyen propre de croiser un agrégat de reclamations avec un comptage de clients sans créer de relation supplémentaire.

## 4.8 Dossier `_Helpers` — couleurs dynamiques (4 mesures)

```dax
Color Statut = 
SWITCH(SELECTEDVALUE(clients[statut]),
    "actif",   "#1D9E75",
    "resilie", "#E24B4A",
    "#888780"
)

Color Offre Churn = 
VAR _t = [Taux Churn Offre %]
RETURN
SWITCH(TRUE(),
    _t >= 30, "#E24B4A",   -- rouge danger
    _t >= 25, "#FFB547",   -- orange warning
    "#1D9E75"              -- vert OK
)

Color Segment RFM = 
SWITCH(SELECTEDVALUE(clients_rfm[segment_rfm]),
    "Champions", "#1D9E75",
    "Loyal",     "#534AB7",
    "At Risk",   "#FFB547",
    "Lost",      "#E24B4A",
    "New",       "#7891B5",
    "#888780"
)

Color Bandeau Alerte = 
IF([Nb Clients 2 Plus Tickets Non Resolus] > 0, "#3D1421", "#0F2C20")
```

### 🎓 MÉTHODE — appliquer une couleur dynamique

Sur un visuel (carte, barre, table) :
1. Format → **Couleur de remplissage** (ou Couleur du texte) → **fx**
2. **Mettre en forme par : Valeur du champ**
3. Choisir la mesure couleur (`[Color Offre Churn]`, etc.)

La cellule prend la couleur retournée par la mesure selon le contexte de ligne.

---
# V — Design system

## 5.1 Charte Customer Churn — Orange chaleureux

| Rôle | Hex | Usage |
|---|---|---|
| Fond page | `#F9F9F8` | Fond général (light, business friendly) |
| Card background | `#FFFFFF` | Cards et tableaux |
| Primaire (orange) | `#E94E1B` | Logo, titres, accents principaux |
| Sidebar foncé | `#A8350E` | Bandeau supérieur, navbar active |
| Vert OK / Champions | `#1D9E75` | Conforme, segment Champions |
| Orange warning | `#FFB547` | Alertes intermédiaires, At Risk |
| Rouge danger | `#E24B4A` | Churn, Lost, alertes critiques |
| Violet RFM Loyal | `#534AB7` | Segment Loyal |
| Texte principal | `#2C2C2A` | Titres, valeurs |
| Texte secondaire | `#888780` | Labels, sous-titres |
| Bordure subtile | `#E5DDD5` | Séparation cards |

### Typographie

| Élément | Police | Taille | Poids |
|---|---|---|---|
| Titre de page | Georgia | 36 | 700 |
| Sous-titre | Segoe UI | 13 | 300 |
| Hero KPI | Segoe UI | 32 | 700 |
| Label KPI | Segoe UI | 12 | 400 (gris) |
| Navbar item | Segoe UI | 13 | 500 |

## 5.2 Mockup PowerPoint → fonds PNG d'arrière-plan

### 📘 Concept clé

Power BI gère mal les arrière-plans complexes. Méthode pro : dessiner dans **PowerPoint** (mockup vierge), exporter en **PNG haute résolution** (1280×720), importer comme **arrière-plan de page**, poser les visuels Power BI **par-dessus**.

### Ce que le mockup PPTX doit contenir

✅ Logo IvoirCom (carré orange `#E94E1B` avec icône abonné blanc) en haut-gauche, navbar horizontale en haut, footer DataProjectLab orange foncé, cards `#FFFFFF` avec ombre subtile et bordure `#E5DDD5`.

❌ Pas de titre de page, pas de slicers, pas de KPI valeurs, pas de chart data.

### 🔧 Méthode 1 — Export PNG depuis PowerPoint à 150 DPI

1. **Win + R** → `regedit` → `HKEY_CURRENT_USER\Software\Microsoft\Office\16.0\PowerPoint\Options`
2. Clic droit → **Nouveau** → **Valeur DWORD (32 bits)** → Nom : `ExportBitmapResolution`, Valeur : `150`
3. Redémarrer PowerPoint, **Fichier → Enregistrer sous → PNG → Toutes les diapositives**

### 🔧 Méthode 2 — CloudConvert

[cloudconvert.com/pptx-to-png](https://cloudconvert.com/pptx-to-png) → upload `mockup_customer_churn_blank.pptx` → 150 DPI → 1280×720.

### Renommage final

```
bg-01-vue-executive.png
bg-02-segments.png
bg-03-cohortes.png
bg-04-reclamations.png
bg-05-plan-action.png
```

### 🔧 Application dans Power BI

1. Sélectionner la page → **Format de la page**
2. **Arrière-plan de la page** → **Ajouter une image** → choisir le PNG
3. **Ajustement** → **Adapter** · **Transparence** → **0 %**

---
# VI — Construire les 5 pages

Cette partie détaille **chaque visuel** avec sa configuration exacte (type, axes, couleurs, étiquettes) et les **méthodes Power BI** non-triviales nécessaires pour le rendu final.

## 6.1 Page 1 — Vue Executive

> *« Quelle est la santé globale de la base abonnés IvoirCom ? »*

**1. Bandeau alerte signal**
- Type : Carte avec valeur dynamique
- Texte : `[Bandeau Alerte]` (ex: « ⚠ 1 195 clients à 2+ tickets non résolus — Taux de churn observé 93,4 % »)
- Fond : `[Color Bandeau Alerte]` via fx → Valeur du champ
- Bordure gauche : 4px rouge `#E24B4A`

**2-5. 4 KPI cards**
- KPI 1 — **Total Clients** : icône 👥 orange, valeur `[Total Clients]` (8 000) blanc 32pt
- KPI 2 — **Taux Churn** : icône ⚠️ rouge, valeur `[Taux Churn %]` formaté `0,0\"%\"` (25,4 %), couleur `#E24B4A` 32pt
- KPI 3 — **ARPU Actifs** : icône 💰 vert, valeur `[ARPU Actifs]` formaté `#,##0\" FCFA\"` (7 959 FCFA) 32pt
- KPI 4 — **Total Réclamations** : icône 📋 orange warning, valeur `[Total Reclamations]` (9 791) 32pt

**6. Donut — Répartition Actifs / Résiliés**
- Type : Anneau
- Catégorie : `clients[statut]`
- Valeur : `[Total Clients]`
- Couleurs : actif vert `#1D9E75`, resilie rouge `#E24B4A`
- Trou central : 65 %

**7. Évolution mensuelle Souscriptions vs Résiliations**
- Type : **Graphique en courbes** (2 séries)
- Axe X : `Calendrier[Annee_Mois]`
- Valeurs : `[Souscriptions Mensuelles]` (vert `#1D9E75`) et `[Resiliations Mensuelles]` (rouge `#E24B4A`)
- Marqueurs : ronds taille 5
- Légende : en bas

**8. Taux de churn par offre**
- Type : Barres horizontales
- Axe Y : `offres[libelle]`
- Axe X : `[Taux Churn Offre %]`
- Couleur barres : `[Color Offre Churn]` via fx → Valeur du champ
- Étiquettes : valeur en `%` à droite des barres
- Tri : descendant

> 🎓 **MÉTHODE — KPI card avec icône**
>
> Power BI ne permet pas d'icône native dans une carte. Astuce :
>
> 1. Insérer une **Forme** (cercle ou carré arrondi) avec la couleur souhaitée.
> 2. Insérer un **Texte** avec l'emoji (📋, ⚠️, 💰).
> 3. Empiler la carte de valeur par-dessus.

## 6.2 Page 2 — Segments à risque

> *« Qui churne le plus (par offre, ville, âge) ? »*

**1-3. 3 KPI cards**
- KPI 1 — **Offres au-dessus seuil 25 %** : valeur `[Nb Offres Au Dessus Seuil]` (2 : Pulse + Étudiant) rouge 32pt
- KPI 2 — **Top ville à risque** : `[Top Ville Churn]` (« Korhogo (27,8 %) ») 24pt
- KPI 3 — **Écart ARPU actifs / churners** : valeur calculée `+71,7 %` 32pt orange

**4. Taux churn par offre (barres horizontales)**
- Identique au visuel 8 de la page 1, mais en plus grand
- Ligne pointillée à 25 % (seuil critique) en `#FFB547`

**5. Taux churn par ville (barres horizontales)**
- Type : Barres horizontales
- Axe Y : `clients[ville]`
- Axe X : `[Taux Churn Ville %]`
- Couleur : `#534AB7` uniforme
- Étiquettes : valeur en %

**6. Heatmap croisée Offre × Ville**
- Type : **Matrix**
- Lignes : `clients[ville]`
- Colonnes : `offres[code_offre]`
- Valeurs : `[Taux Churn Offre %]`
- Mise en forme conditionnelle : **Mettre en échelle des couleurs** sur valeurs (vert `#1D9E75` à 0, rouge `#E24B4A` à 50)

> 🎓 **MÉTHODE — Heatmap dans une matrice Power BI**
>
> 1. Visuel **Matrix**
> 2. Format → **Éléments de cellule** → Activer **Couleur d'arrière-plan**
> 3. fx → **Mise en échelle des couleurs** → Valeurs : Min `#1D9E75`, Centre `#FFB547` (à 25), Max `#E24B4A`
> 4. Format → **Total général** : Désactiver pour ne pas polluer la heatmap

**7. Taux churn par tranche d'âge (barres verticales)**
- Axe X : `clients[Tranche_Age]` (trié alpha pour avoir 18-25 → 60+)
- Axe Y : `[Taux Churn Tranche Age %]`
- Couleur : orange `#E94E1B`

## 6.3 Page 3 — Cohortes & Rétention

> *« Quelle génération de souscripteurs tient le mieux ? »*

**1-3. 3 KPI cards**
- KPI 1 — **Ancienneté actifs** : `[Anciennete Actifs Mois]` (25,9 mois) vert 32pt
- KPI 2 — **Ancienneté churners** : `[Anciennete Churners Mois]` (17,2 mois) rouge 32pt
- KPI 3 — **Écart actifs / churners** : valeur calculée `+8,7 mois` orange 24pt

**4. Heatmap rétention 12 cohortes × 12 mois**
- Type : **Matrix**
- Lignes : `clients[Cohorte_Mois]` (formaté yyyy-MM, filtré 2024)
- Colonnes : `clients[Anciennete_Mois]` (filtré 0 à 11)
- Valeurs : `[Retention Cohorte Pct]`
- Format conditionnel : Min `#E24B4A` à 50 %, Centre `#FFB547` à 80 %, Max `#1D9E75` à 100 %
- Format → Total général : Désactivé

**5. Courbe rétention médiane**
- Type : Graphique en courbes
- Axe X : `clients[Anciennete_Mois]` (0 à 11)
- Axe Y : `[Retention Cohorte Pct]` (avec `MEDIAN` à la place via mesure dédiée si besoin)
- Couleur ligne : orange `#E94E1B` épaisseur 2px

**6. Table — Top 5 cohortes les plus solides / fragiles**
- Colonnes : `Cohorte_Mois`, taille initiale, rétention M+11, écart vs médiane
- Tri : descendant sur rétention M+11

> 🎓 **MÉTHODE — Filtrer une matrice à 12 mois × 12 cohortes**
>
> Sur le panneau **Filtres** du visuel Matrix :
> 1. `clients[Cohorte_Mois]` → filtrer entre `2024-01-01` et `2024-12-31`
> 2. `clients[Anciennete_Mois]` → filtrer **est inférieur ou égal à** `11`
> 3. Sans ces filtres, la matrice essaie de tracer toutes les combinaisons et plante.

## 6.4 Page 4 — Signaux Réclamations

> *« Les plaintes prédisent-elles le départ ? »*

**1-3. 3 KPI cards**
- KPI 1 — **Total Réclamations** : `[Total Reclamations]` (9 791) orange 32pt
- KPI 2 — **% Tickets Non Résolus** : `[Pct Tickets Non Resolus]` (47,8 %) rouge 32pt
- KPI 3 — **Délai Résolution Moyen** : `[Delai Resolution Moyen Jours]` (~4 j) bleu 32pt

**4. Barres empilées Type × Statut**
- Type : Histogramme empilé
- Axe X : `reclamations[type]` (Reseau, Facturation, Offre, Materiel)
- Légende : `reclamations[statut]` (resolu, ouvert, abandonne)
- Valeur : `COUNTROWS(reclamations)`
- Couleurs : resolu vert `#1D9E75`, ouvert orange `#FFB547`, abandonne rouge `#E24B4A`

**5. Comparaison tickets moyens Actifs vs Churners**
- Type : Barres horizontales
- 2 mesures : `[Tickets Moyens Actifs]` (~0,78) et `[Tickets Moyens Churners]` (~2,53)
- Couleurs respectives vert et rouge
- Étiquettes : valeurs avec 2 décimales

**6. Bandeau hero — Signal phare**
- Carte large bordée rouge
- Texte hardcodé : « 93,4 % de churn parmi les clients à 2+ tickets non résolus » (vs 25,4 % base globale)
- Sous-texte : `[Bandeau Alerte]`

> 🎓 **MÉTHODE — Bandeau hero avec 2 valeurs côte à côte**
>
> 1. Insérer une **Forme rectangle** (fond `#FFF4ED`, bordure 2px `#E24B4A`)
> 2. Superposer 2 **Cartes** : à gauche `[Taux Churn Segment Risque %]` (formaté `0,0\"%\"`), à droite `[Taux Churn %]`
> 3. Titre au-dessus : « Risque vs Base globale »
> 4. Le contraste visuel (93,4 % vs 25,4 %) raconte l'histoire en 1 seconde

**7. Top 10 clients à plus de tickets non résolus**
- Type : Table
- Colonnes : `id_client`, `code_offre`, `ville`, nb tickets non résolus, ARPU 6m
- Tri : descendant nb tickets
- Couleur conditionnelle : ARPU > 10 000 FCFA en rouge (= valeur perdue prioritaire)

## 6.5 Page 5 — Plan d'Action

> *« Quels clients appeler aujourd'hui ? »*

**1-3. 3 KPI cards**
- KPI 1 — **Champions** : `[Nb Clients Champions]` (676) vert 32pt + sous-texte « 8,5 % de la base »
- KPI 2 — **At Risk** : `[Nb Clients At Risk]` (1 266) orange 32pt + « 15,8 % »
- KPI 3 — **Lost** : `[Nb Clients Lost]` (1 934) rouge 32pt + « 24,2 % »

**4. Donut — Répartition par segment RFM**
- Catégorie : `clients_rfm[segment_rfm]`
- Valeur : `COUNTROWS(clients_rfm)`
- Couleurs via `[Color Segment RFM]`
- Trou central : 65 %
- Légende : en bas

**5. Barres horizontales — Taux churn par segment RFM**
- Axe Y : `clients_rfm[segment_rfm]`
- Axe X : taux churn calculé par segment via mesure (Champions 0,1 % / Loyal 1,3 % / At Risk 20,5 % / Lost 82,2 %)
- Couleur : `[Color Segment RFM]`

**6. Scatter — Récence × ARPU 6m**
- Type : Nuage de points
- Détails : `clients_rfm[id_client]`
- Axe X : `clients_rfm[jours_depuis_derniere_facture]`
- Axe Y : `clients_rfm[arpu_6m]`
- Taille : constante (ou `[Total Clients]` si tu veux densifier)
- Couleur : `[Color Segment RFM]`
- Sous-titre : « Quadrant supérieur-gauche = ARPU haut + récence faible = Champions »

**7. Table — Top 50 At Risk**
- Filtre visuel : `clients_rfm[segment_rfm] = "At Risk"`
- Top N : 50 par `arpu_6m DESC`
- Colonnes : `id_client`, `code_offre`, `ville`, `jours_depuis_derniere_facture`, `arpu_6m`, R/F/M scores
- Mise en forme : ARPU > 25 000 → fond rouge pâle

**8. Card hero — ARPU annuel exposé**
- Valeur : `[ARPU Top 50 At Risk]` (~15,6 M FCFA)
- Sous-texte : « ARPU annuel des Top 50 At Risk si on ne fait rien »

> 🎓 **MÉTHODE — Filtrer Top N dans un visuel Power BI**
>
> Sur le panneau **Filtres** du visuel Table :
> 1. Glisser `clients_rfm[id_client]` dans la zone **Filtres au niveau visuel**
> 2. Type de filtre : **Top N**
> 3. Afficher : **N supérieur** = `50`
> 4. Par valeur : `clients_rfm[arpu_6m]` (Somme)
> 5. Combiner avec un autre filtre `segment_rfm = "At Risk"` pour cibler

---
# VII — Slicers, navigation, finitions

**Navigation horizontale en haut de chaque page** : 5 textes cliquables « Vue Executive · Segments · Cohortes · Réclamations · Plan d'action ». Item actif : texte orange `#E94E1B` avec soulignement 2px ; items inactifs : texte gris `#888780`.

> 🎓 **MÉTHODE — Navbar horizontale avec état actif**
>
> Power BI ne gère pas l'état actif natif. Astuce :
>
> 1. Sur chaque page, créer 5 boutons **Texte** (un par section), tous avec **action Navigation de page**
> 2. Sur la page courante, le bouton correspondant à la page active n'a PAS d'action de navigation et utilise un style différent (orange + soulignement)
> 3. Dupliquer la disposition des 5 boutons sur les 5 pages, en variant à chaque fois le bouton actif

**Slicer global Année** : 1 slicer déroulant `Calendrier[Annee]` (« Tout / 2024 / 2025 ») en haut à droite. Synchroniser sur toutes les pages : Format → **Synchroniser les segments → cocher toutes les pages**.

**Slicer secondaire Ville** (page 2 uniquement) : `clients[ville]` en bouton segmenté (Format → Style → Vignette).

**Logo « IvoirCom »** : carré orange `#E94E1B` avec icône abonné blanc en haut-gauche, suivi du nom **IvoirCom** blanc 16pt et sous-titre « Customer Churn Analytics » gris clair 11pt.

---
# VIII — Validation et livraison

## 8.1 Checklist de recette

**Modèle** : 6 tables sources + Calendrier + _Mesures, 8 relations actives + 2 inactives sur clients (date_souscription, date_resiliation), 0 LocalDateTable · `Calendrier` marquée comme table de dates · Auto Date/Time désactivé.

**Mesures** : ~50 dans `_Mesures` · 7 dossiers numérotés `0.` à `6.` + `_Helpers` · format défini (FCFA, %, mois, ratio).

**Pages** : 5 pages avec sous-titre dynamique · Slicer Année synchronisé · Navbar horizontale avec état actif · Couleurs conformes à la charte Orange chaleureux.

**Performance** : ouverture < 5 s · aucun visuel en erreur.

**Valeurs cibles à vérifier visuellement** :

| Visuel | Valeur attendue |
|---|---|
| KPI Total Clients | 8 000 |
| KPI Taux Churn | 25,4 % |
| KPI ARPU Actifs | 7 959 FCFA |
| KPI Total Réclamations | 9 791 |
| Donut Actifs/Résiliés | 5 967 / 2 033 |
| Top offre churn | Pulse 35,3 % |
| Top ville churn | Korhogo 27,8 % |
| Bandeau alerte signal | 1 195 clients à 2+ tickets non résolus |
| Taux churn segment risque | 93,4 % |
| Nb Champions | 676 (8,5 %) |
| Nb At Risk | 1 266 (15,8 %) |
| ARPU Top 50 At Risk annuel | ~15,6 M FCFA |

## 8.2 Pièges fréquents

| Symptôme | Cause | Correction |
|---|---|---|
| Souscriptions Mensuelles toujours = 0 | Relation Calendrier ↔ clients[date_souscription] est inactive et `USERELATIONSHIP` oublié | Wrapper la mesure dans `CALCULATE(..., USERELATIONSHIP(clients[date_souscription], Calendrier[Date]))` |
| `RANKX` retourne 1 partout dans une table | Pas de transition de contexte ligne → filtre | Wrapper la mesure dans `CALCULATE([Mesure])` à l'intérieur du RANKX |
| `DATEDIFF` renvoie blank | `Calendrier` non marquée comme table de dates | Outils de table → Marquer comme table de dates |
| Heatmap rétention vide | Filtre `Anciennete_Mois <= 11` non appliqué → matrice essaie 100+ colonnes | Appliquer filtre visuel sur `Anciennete_Mois` BETWEEN 0 AND 11 |
| Donut Actifs/Résiliés affiche 8 030 | Doublons `_DUP` non exclus dans Power Query | Filtre Power Query : `id_client doesn't end with "_DUP"` |
| ARPU = 0 sur certains clients | Factures à montant 0 incluses | Mesure `ARPU` doit `FILTER(factures, factures[montant_fcfa] > 0)` |
| Bouton actif navbar ne change pas selon la page | Power BI ne gère pas l'état actif natif | Dupliquer le bouton avec le style actif sur chaque page |

## 8.3 Storytelling exécutif

Pour présenter au directeur commercial IvoirCom, suis l'ordre des 5 pages :

1. **Vue Executive** : « 8 000 abonnés, taux de churn cumulatif 25,4 % (au-dessus cible 20 %). ARPU actifs 7 959 FCFA, ARPU churners 4 636 FCFA — soit +71,7 % chez les actifs. 9 791 réclamations sur la période, 47,8 % non résolues. »
2. **Segments à risque** : « Pulse 35,3 % et Étudiant 31,4 % concentrent 61 % du churn total. Toutes les villes sont autour de 25-28 % — le problème n'est pas géographique mais par offre. Cellule la plus rouge : Pulse à Korhogo. »
3. **Cohortes & Rétention** : « Rétention M+11 ~92 % sur la cohorte janvier 2024 — l'onboarding fonctionne. La dégradation est lente et régulière, donc structurelle (concurrence Orange/MTN/Moov). »
4. **Signaux Réclamations** : « 71,5 % des churners ont 2+ tickets vs 17,9 % des actifs (ratio x4). Sur les 1 195 clients à 2+ tickets non résolus, 93,4 % churnent — c'est le signal le plus fort de l'analyse. »
5. **Plan d'action** : « Seulement 8,5 % de Champions (cible 15-25 %), 24,2 % de Lost. 1 266 At Risk dont 50 abonnés Pro à 25 000 FCFA mensuels = 15,6 M FCFA d'ARPU annuel exposé. »
6. **Recommandation** : 3 leviers chiffrés activables dès la semaine prochaine, sans budget ML — total ~75 à 85 M FCFA d'ARPU récupéré / défendu par an.

## 8.4 Annexes — Mapping mockup PPTX ↔ pages Power BI

| Slide | Background PNG | Page |
|---|---|---|
| 1 | `bg-01-vue-executive.png` | Vue Executive |
| 2 | `bg-02-segments.png` | Segments à risque |
| 3 | `bg-03-cohortes.png` | Cohortes & Rétention |
| 4 | `bg-04-reclamations.png` | Signaux Réclamations |
| 5 | `bg-05-plan-action.png` | Plan d'action |

### Livrables finaux

- `Rapport-customer-churn.pbix`
- `mockup_customer_churn.pptx`
- 5 fichiers `bg-XX-*.png`
- `synthese_customer_churn.pptx` (12 slides, optionnel)
- `recommandations_churn.docx`

---
<div style="background:#A8350E;padding:24px 32px;border-radius:10px;color:#FFFFFF;font-family:Georgia,serif;text-align:center;">
<div style="font-size:22px;font-weight:700;margin-bottom:6px;">Customer Churn Analytics — IvoirCom</div>
<div style="font-size:13px;color:#FFE4D6;font-family:'Segoe UI',sans-serif;"><b>DataProjectLab</b> — apprendre la data sur des cas concrets, structurés et orientés métier.</div>
</div>